In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🕵️ Competitive Intelligence Crew

## What We're Building

A 4-agent competitive intelligence pipeline:

| Agent | Role |
|-------|------|
| **Competitor Tracker** | Monitor competitor activities and announcements |
| **Launch Summarizer** | Summarize product launches and updates |
| **Feature/Price Extractor** | Extract pricing tiers and feature matrices |
| **Memo Writer** | Compile a weekly competitive intelligence memo |

## Key CrewAI Concept: `kickoff_for_each()`

We can run the same crew for multiple competitors using `kickoff_for_each()`,
which iterates the crew over a list of inputs.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Competitor Data

In [ ]:
our_company = "DataSync Pro — B2B data integration platform"

# Simulated competitor intelligence (in production, this would come from web scraping)
competitor_updates = """
COMPETITOR UPDATE FEED (past 2 weeks):

--- COMPETITOR 1: Fivetran ---
- Announced SOC 2 Type II certification for all connectors
- Launched 15 new connectors including Notion, Linear, and Airtable
- Raised pricing: Starter plan now $500/mo (was $400/mo)
- New feature: "QuickStart" templates for common data pipelines
- Blog post: "Why ETL is dead — long live ELT" (2,500 shares)
- Hired VP of Enterprise Sales from Snowflake

--- COMPETITOR 2: Airbyte ---
- Released Airbyte Cloud 2.0 with AI-powered schema mapping
- Open-source repo hit 15,000 GitHub stars
- Announced $200M Series C at $1.5B valuation
- New feature: "Connector Builder" — lets users create custom connectors in 10 min
- Dropped prices 20% across all tiers
- Launched developer certification program

--- COMPETITOR 3: Stitch (by Talend) ---
- Quiet period — no major product announcements
- Reduced marketing spend (fewer blog posts, social activity down 40%)
- Lost 2 senior engineers to Airbyte (LinkedIn confirms)
- Still advertising at old pricing ($100/mo starter)
- Talend parent company reported flat Q3 revenue
"""

print("Competitor update feed loaded (3 competitors)")

## Step 3 — Create Agents

In [ ]:
tracker = Agent(
    role="Competitive Intelligence Tracker",
    goal="Monitor and categorize all competitor activities by strategic importance",
    backstory=(
        "You are a competitive intelligence analyst at a B2B SaaS company. "
        "You track competitors daily across press releases, blogs, social media, "
        "job postings, and GitHub activity. You know that hiring patterns and "
        "pricing changes are often more revealing than product announcements."
    ),
    llm=llm,
    verbose=True,
)

launch_summarizer = Agent(
    role="Product Launch Analyst",
    goal="Summarize competitor product launches and assess their impact",
    backstory=(
        "You analyze product launches for a living. You assess each launch on: "
        "novelty (is this truly new?), threat level (does this eat our lunch?), "
        "and customer impact (will our customers care?). You're not easily impressed."
    ),
    llm=llm,
    verbose=True,
)

feature_extractor = Agent(
    role="Pricing & Feature Analyst",
    goal="Extract and compare pricing tiers, features, and positioning changes",
    backstory=(
        "You build competitive feature matrices for sales teams. You compare "
        "pricing, features, integrations, and support levels across competitors. "
        "You highlight where we win, where we lose, and where it's a tie."
    ),
    llm=llm,
    verbose=True,
)

memo_writer = Agent(
    role="Weekly Intel Memo Writer",
    goal="Compile a concise, actionable weekly competitive intelligence memo",
    backstory=(
        "You write the weekly competitor briefing that goes to the CEO, VP Sales, "
        "and VP Product. It must be scannable in 2 minutes, highlight what changed, "
        "and end with clear action items. No rambling."
    ),
    llm=llm,
    verbose=True,
)

print("4 agents created")

## Step 4 — Create Tasks

In [ ]:
track_task = Task(
    description=f"""Analyze this competitor update feed and categorize each activity:

Our company: {our_company}

{competitor_updates}

For each competitor, categorize activities as:
- 🔴 HIGH THREAT: Directly threatens our business
- 🟡 MEDIUM: Worth watching, not immediate threat
- 🟢 LOW: Minor activity, no action needed
- 💡 OPPORTUNITY: Something we can exploit

Also identify any SIGNALS:
- Is a competitor struggling (layoffs, quiet period)?
- Is a competitor gaining momentum (funding, hiring)?
- Are there market shifts affecting everyone?""",
    expected_output="Categorized competitor activity feed with threat levels.",
    agent=tracker,
)

launch_task = Task(
    description="""For each product launch or feature announcement identified above:

1. WHAT: One-sentence description of the launch
2. SO WHAT: Why should we care? Impact on our market position
3. NOW WHAT: Recommended response (ignore, monitor, counter, copy)
4. THREAT SCORE: 1-10 (10 = existential threat)
5. TIMELINE: How urgent is our response?""",
    expected_output="Launch impact assessments with threat scores and response recs.",
    agent=launch_summarizer,
)

feature_task = Task(
    description=f"""Create an updated competitive comparison matrix:

Our product: {our_company}

Build a feature/pricing comparison table:
1. Pricing tiers: Starter / Growth / Enterprise for each competitor
2. Key features: connectors, setup time, schema handling, security
3. For each row, mark: ✅ We win | ❌ They win | ➡️ Tie
4. Note any recent changes to competitor pricing or features
5. Identify our strongest differentiators and biggest gaps""",
    expected_output="Competitive comparison matrix with win/loss analysis.",
    agent=feature_extractor,
)

memo_task = Task(
    description="""Write the weekly competitive intelligence memo.

Format:
## COMPETITIVE INTELLIGENCE — WEEKLY MEMO

### Bottom Line Up Front (BLUF)
(3 sentences: What happened, what matters, what to do)

### Key Moves This Week
(Bullet list of top 5 activities, ranked by importance)

### Competitor Health Check
(One-line status for each competitor: 🟢 Growing | 🟡 Stable | 🔴 Declining)

### Feature/Price Watch
(Summary of any pricing or feature changes)

### Recommended Actions
(3-5 specific, actionable items for our team)

Keep the entire memo under 500 words.""",
    expected_output="A concise weekly competitive intelligence memo.",
    agent=memo_writer,
    markdown=True,
)

print("4 tasks created")

## Step 5 — Run the Crew

In [ ]:
intel_crew = Crew(
    agents=[tracker, launch_summarizer, feature_extractor, memo_writer],
    tasks=[track_task, launch_task, feature_task, memo_task],
    process=Process.sequential,
    verbose=True,
)

print("Competitive intelligence crew assembled!")
result = intel_crew.kickoff()
print("\n✅ Weekly memo generated!")

In [ ]:
print("📊 WEEKLY COMPETITIVE INTELLIGENCE MEMO:")
print("=" * 60)
print(result.raw)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Threat categorization** | 🔴/🟡/🟢/💡 severity levels for competitor moves |
| **BLUF format** | Bottom Line Up Front — executives read this first |
| **What/So What/Now What** | Framework for assessing competitive impact |
| **Feature matrix** | Win/loss comparison for sales enablement |

## 🔧 Extensions

- **Web scraping**: Use tools to monitor competitor websites automatically
- **kickoff_for_each()**: Run the crew weekly with fresh competitor data
- **Slack integration**: Auto-post the memo to a Slack channel
- **Trend tracking**: Store memos and track competitor trajectories over time